# Lesson 03 — Building makemore Part 2: MLP

- **GitHub issue:** [#3](https://github.com/majorgilles/karpathy_ml_course/issues/3)
- **Video:** https://youtu.be/TCH_1BHY58I
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Load names and prepare the MLP workspace

This lesson moves beyond a bigram model: the MLP will use a fixed number of previous characters, called a **context window**, to predict the next character. First load the names as Python strings and establish the small set of libraries used for tensors, one-hot encoding, and later visualizations.

Each element of `words` is one name. The early cells inspect a few names and the dataset size before converting characters into numeric model inputs.


In [1]:
import torch  # Tensor operations and model parameters.
import torch.nn.functional as F  # One-hot encoding and other neural-network helpers.
import matplotlib.pyplot as plt  # Visualizations used later in the lesson.
from sympy.codegen.ast import float32  # Current exploration import; not used by these cells yet.
%matplotlib inline

In [2]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 2. Map characters to model-friendly IDs

Neural networks work with numbers, not Python characters. `stoi` means string-to-index and maps each character to one stable integer; `itos` reverses that lookup for readable examples and generated output.

The boundary token `.` receives index `0`. It represents both left padding before a name and the end of a name, so the context window can start before any real letters have appeared.


In [4]:
# Collect the 26 lowercase letters once and sort them for reproducible IDs.
chars = sorted(list(set(''.join(words))))
# Reserve 0 for the boundary token, so letters begin at index 1.
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# Invert the mapping: model indices back to printable characters.
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 3. Turn names into fixed-context training examples

`block_size = 3` means every input row contains exactly three previous character IDs. `X` stores those three-character contexts and `Y` stores the one next-character target that followed each context.

For `emma`, the first example is `... → e`, then `..e → m`, and so on until `mma → .`. The initial three zeros are left padding with the boundary token. `words[:5]` deliberately keeps this printed walkthrough small; use all of `words` when building the full training dataset.


In [5]:
# Build (context, next-character) examples from five names for an inspectable walkthrough.
BLOCK_SIZE = 3  # Number of preceding characters the model receives as input.
X, Y = [], []  # X holds context rows; Y holds one next-character target per row.

for w in words[:5]:  # Use all `words` later for the full training dataset.
    print(w)
    context = [0] * BLOCK_SIZE  # Start with three boundary-token IDs: "...".

    for ch in w + ".":  # Include the final boundary token as a target.
        ix = stoi[ch]  # Integer ID of the character this context should predict.
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), "--->", itos[ix])
        context = context[1:] + [ix]  # Drop the oldest ID and append this target ID.

# Convert Python lists to integer tensors for embedding lookup in the MLP.
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
# Verify: one three-ID context per example and one integer next-character target.
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

## 4. Look up embeddings for individual tokens, lists, and whole contexts

`C` is an embedding table with one row per vocabulary token. Each row contains two learnable features in this small example. Integer indexing retrieves rows from that table: one ID gives one vector, a list of IDs gives several vectors, and a tensor of context IDs gives an embedding vector for every token in every context.

The important input/output relationship is `C[X]`: `X` contains 32 examples with 3 token IDs each, so the lookup replaces every ID with its two-feature embedding. The result has shape `(32, 3, 2)`: examples, context positions, embedding features.


In [7]:
# Create a 27-row embedding table; each token initially receives two random features.
EMBEDDING_SIZE = 2
VOCAB_SIZE = 27
C = torch.randn((VOCAB_SIZE, EMBEDDING_SIZE))

In [8]:
# Direct lookup: one token ID selects one embedding row with two features.
C[5]

tensor([-0.9512,  1.0038])

In [9]:
# A list of IDs selects several rows, preserving the list order: output shape (3, 2).
C[[5, 6, 7]]

tensor([[-0.9512,  1.0038],
        [ 1.7675, -0.7174],
        [-1.3907, -1.0977]])

In [10]:
# Batched lookup: replace every ID in X with its C row, giving shape (32, 3, 2).
emb = C[X]
emb

tensor([[[-1.0834, -0.1978],
         [-1.0834, -0.1978],
         [-1.0834, -0.1978]],

        [[-1.0834, -0.1978],
         [-1.0834, -0.1978],
         [-0.9512,  1.0038]],

        [[-1.0834, -0.1978],
         [-0.9512,  1.0038],
         [ 0.4393,  0.4169]],

        [[-0.9512,  1.0038],
         [ 0.4393,  0.4169],
         [ 0.4393,  0.4169]],

        [[ 0.4393,  0.4169],
         [ 0.4393,  0.4169],
         [ 0.1662, -0.2429]],

        [[-1.0834, -0.1978],
         [-1.0834, -0.1978],
         [-1.0834, -0.1978]],

        [[-1.0834, -0.1978],
         [-1.0834, -0.1978],
         [ 1.0580, -0.0109]],

        [[-1.0834, -0.1978],
         [ 1.0580, -0.0109],
         [-0.4317,  1.4370]],

        [[ 1.0580, -0.0109],
         [-0.4317,  1.4370],
         [ 0.9269,  0.4242]],

        [[-0.4317,  1.4370],
         [ 0.9269,  0.4242],
         [-0.0074, -0.1387]],

        [[ 0.9269,  0.4242],
         [-0.0074, -0.1387],
         [ 0.9269,  0.4242]],

        [[-0.0074, -0

In [11]:
# Inspect the three axes: training examples, context positions, and embedding features.
emb.shape

torch.Size([32, 3, 2])

## 5. Flatten each context into one MLP input row

`emb` has shape `(32, 3, 2)`: 32 training examples, 3 context positions, and 2 embedding features per position. The first linear layer expects one feature vector per example, so reshape combines the last two dimensions into six features while preserving the batch dimension.

`emb.reshape(-1, EMBEDDING_SIZE * BLOCK_SIZE)` therefore produces shape `(32, 6)`. The `-1` tells PyTorch to infer the number of examples. The `unbind` plus `cat` expression is an equivalent demonstration: split the three context positions into three `(32, 2)` tensors, then concatenate them into `(32, 6)`.

The comparison with `==` is elementwise, so it displays a `(32, 6)` grid of `True` values. `torch.equal` would instead return one Boolean answer for the entire tensors.


In [12]:
# First linear layer: six flattened context features feed 100 hidden neurons.
FIRST_HIDDEN_UNITS = 100
w1 = torch.randn((BLOCK_SIZE * EMBEDDING_SIZE, FIRST_HIDDEN_UNITS))
b1 = torch.randn(FIRST_HIDDEN_UNITS)  # One bias value per hidden neuron.

In [13]:
# Keep each example row and flatten its 3 × 2 context embedding into six MLP features.
reshaped = emb.reshape(-1, EMBEDDING_SIZE * BLOCK_SIZE)
# Elementwise check: every entry should be True because split-then-concatenate gives the same layout.
reshaped == torch.cat(torch.unbind(emb, dim=1), dim=1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

## 6. Produce hidden activations with a linear layer and `tanh`

Matrix multiplication combines each six-feature context row with `w1`: `(32, 6) @ (6, 100) → (32, 100)`. Adding `b1`, which has shape `(100,)`, broadcasts one bias per hidden neuron across all 32 examples.

$$
h = \tanh(\operatorname{flattened\_context} W_1 + b_1)
$$

Here, `h` is the `(32, 100)` hidden-activation tensor, `W_1` is `w1`, and `b_1` is `b1`. `tanh` applies to every raw pre-activation score and maps it into the range from `-1` to `1`. This nonlinearity matters because stacked linear layers alone could be reduced to one linear transformation; `tanh` lets the MLP represent more complex relationships between context characters and the next target.


In [14]:
# Inspect the input, weight, and matrix-product shapes of the first linear layer.
print(reshaped.shape)
print(w1.shape)
print((reshaped @ w1).shape)
# Compute 100 linear pre-activation scores per example, then apply tanh element by element.
h = torch.tanh(reshaped @ w1 + b1)
# Confirm: one 100-feature hidden activation vector for each of the 32 examples.
h.shape

torch.Size([32, 6])
torch.Size([6, 100])
torch.Size([32, 100])


torch.Size([32, 100])

## 7. Map hidden activations to one score per vocabulary candidate

Each of the 32 hidden vectors has 100 features. The output layer maps those features to 27 raw scores, one for every vocabulary candidate: the 26 letters plus the boundary token `.`.

$$
\operatorname{logits} = h W_2 + b_2
$$

`W_2` is `w2` with shape `(100, 27)`, `b_2` is `b2` with shape `(27,)`, and `h` has shape `(32, 100)`. The resulting `logits` tensor has shape `(32, 27)`: one row per training example and one column per candidate target character. Logits are raw preference scores, not probabilities; the next cells convert each row into a probability distribution.


In [15]:
# Output-layer weights map 100 hidden features to one raw score for each vocabulary candidate.
w2 = torch.randn((FIRST_HIDDEN_UNITS, VOCAB_SIZE))
b2 = torch.randn(VOCAB_SIZE)  # One bias value per output vocabulary candidate.

In [16]:
# Compute one raw candidate score per vocabulary token for each training example: (32, 100) @ (100, 27) → (32, 27).
logits = h @ w2 + b2
logits.shape

torch.Size([32, 27])

## 8. Convert logits into probability distributions with manual softmax

`logits` contains 27 raw candidate scores per training example. Exponentiation makes every score positive; these intermediate values are named `counts` by convention here, but they are not observed character counts. Dividing each row by its own total gives 27 probabilities that add to one.

$$
p_{i,j} = \frac{\exp(\ell_{i,j})}{\sum_{k=0}^{26} \exp(\ell_{i,k})}
$$

Here, `i` identifies one of the 32 training examples, `j` identifies one candidate vocabulary token, `k` ranges over all 27 candidates, and `ℓ` is a logit. The expected target for example `i` is `Y[i]`; when the loss is introduced, `probs[i, Y[i]]` is the one probability that directly contributes to that example's negative log-likelihood.


In [17]:
# Exponentiate every raw score to create positive, unnormalized softmax values.
counts = logits.exp()

In [20]:
# Normalize each example's 27 candidate values by its row total; keepdim preserves shape (32, 1) for row-wise division.
probs = counts / counts.sum(1, keepdim=True)
# Verify the output shape and that two example distributions each use the full probability budget of 1.
print(probs.shape)
print(probs[0].sum())
print(probs[1].sum())

torch.Size([32, 27])
tensor(1.)
tensor(1.0000)
